In [ ]:
RANDOM_STATE = 42
MODEL_NAME = 'distilbert-base-uncased'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 8
HEADER_MAX_LEN = 128
BODY_MAX_LEN = 256
EPOCHS = 3
LEARNING_RATE = 2e-5

c:\Users\Jay\Projects\AI-extension-BEC-detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using dataset directory: c:\Users\Jay\Projects\AI-extension-BEC-detection\Dataset
Output directory: c:\Users\Jay\Projects\AI-extension-BEC-detection\outputs\distilbert_header_body
Device: cuda


In [6]:
# =============================================================================
# DATASET BUILD + LABELS + DUAL-ENCODER MODEL SETUP
# =============================================================================
from pathlib import Path

# Configuration for thresholds and verdicts
CFG = {
    'manip_threshold': 0.5,   # Threshold for marking manipulation tactic as active
    'risk_low': 0.4,          # Below this: ALLOW verdict
    'risk_mid': 0.7,          # Above this: BLOCK verdict
}

# Tactics / lexicons
MANIP_LABELS = ['authority', 'fear', 'urgency', 'reward', 'trust']
LEXICONS = {
    'authority': ['ceo', 'manager', 'director', 'supervisor', 'urgent approval', 'verify account'],
    'fear':      ['suspend', 'locked', 'security alert', 'fraud', 'account will be closed'],
    'urgency':   ['urgent', 'immediately', 'asap', 'act now', 'within 24 hours'],
    'reward':    ['winner', 'gift card', 'reward', 'bonus', 'claim your prize'],
    'trust':     ['invoice', 'payment', 'document', 'shared file', 'update your details'],
}

In [ ]:
# =============================================================================
# PHASE 3 — DATA SPLITS + TRAINING  (Dual Parallel Analysis)
# =============================================================================
log.stage("PHASE 3 — DUAL PARALLEL ANALYSIS: Multi-Task Training")

# ── Splits: 70 / 15 / 15 ──────────────────────────────────────────────────────
log.info("Splitting: 70% train | 15% val | 15% zero-day test …")
train_df, tmp_df = train_test_split(df, test_size=0.30,
                                    stratify=df['label'], random_state=RANDOM_STATE)
val_df,  test_df = train_test_split(tmp_df, test_size=0.50,
                                    stratify=tmp_df['label'], random_state=RANDOM_STATE)

log.ok(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  ZD-Test: {len(test_df):,}")
log.metric(f"Train  → Phishing: {train_df['label'].sum():,}  "
           f"Legit: {(train_df['label']==0).sum():,}")
log.metric(f"ZD Test→ Phishing: {test_df['label'].sum():,}  "
           f"Legit: {(test_df['label']==0).sum():,}")

# ── Encoding helpers: separate header and body encodings with different max lengths
def encode_texts(tok, texts, max_len):
    return tok(list(texts), max_length=max_len, padding='max_length', truncation=True, return_tensors='pt')

class PhishDS(Dataset):
    def __init__(self, head_enc, body_enc, labels, manip):
        self.head_enc = head_enc
        self.body_enc = body_enc
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.manip = torch.tensor(manip, dtype=torch.float)

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        return {
            'h_input_ids': self.head_enc['input_ids'][idx],
            'h_attention_mask': self.head_enc['attention_mask'][idx],
            'b_input_ids': self.body_enc['input_ids'][idx],
            'b_attention_mask': self.body_enc['attention_mask'][idx],
            'label': self.labels[idx],
            'manip': self.manip[idx],
        }

# prepare tactic column names
manip_cols = [f'flag_{l}' for l in MANIP_LABELS]

# create separate encodings for header and body (do this after splits)
train_head_enc = encode_texts(tokenizer, train_df['header'].fillna('').astype(str), HEADER_MAX_LEN)
train_body_enc = encode_texts(tokenizer, train_df['body'].fillna('').astype(str), BODY_MAX_LEN)
val_head_enc   = encode_texts(tokenizer, val_df['header'].fillna('').astype(str), HEADER_MAX_LEN)
val_body_enc   = encode_texts(tokenizer, val_df['body'].fillna('').astype(str), BODY_MAX_LEN)
test_head_enc  = encode_texts(tokenizer, test_df['header'].fillna('').astype(str), HEADER_MAX_LEN)
test_body_enc  = encode_texts(tokenizer, test_df['body'].fillna('').astype(str), BODY_MAX_LEN)

train_loader = DataLoader(PhishDS(train_head_enc, train_body_enc, train_df['label'].values, train_df[manip_cols].values), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(PhishDS(val_head_enc,   val_body_enc,   val_df['label'].values,   val_df[manip_cols].values),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(PhishDS(test_head_enc,  test_body_enc,  test_df['label'].values,  test_df[manip_cols].values),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
log.ok(f"DataLoaders — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

# ── Optimiser + Scheduler ──────────────────────────────────────────────────────
optimizer    = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
ce_loss      = nn.CrossEntropyLoss()
bce_loss     = nn.BCELoss()

log.info(f"Optimiser: AdamW  lr={LEARNING_RATE}  wd=0.01")
log.info(f"Scheduler: linear warmup {warmup_steps} steps → decay")
log.info(f"Total steps: {total_steps}")

def mt_loss(logits, labels, mp, mt, anom):
    """Multi-task loss: 0.60·CE + 0.25·BCE_manip + 0.15·BCE_anom"""
    lc = ce_loss(logits, labels)
    lm = bce_loss(mp, mt)
    la = F.binary_cross_entropy(anom, labels.float().unsqueeze(-1))
    return 0.60*lc + 0.25*lm + 0.15*la, lc.item(), lm.item(), la.item()

# ── Training Loop ───────────────────────────────────────────────────────────────
best_f1 = 0.0
history = {'train_loss':[], 'val_loss':[], 'val_f1':[], 'val_auc':[]}

for epoch in range(1, EPOCHS+1):
    # ── TRAIN ──────────────────────────────────────────────────────────────
    log.info(f"EPOCH {epoch}/{EPOCHS} ── Training ({len(train_loader)} batches) …")
    model.train()
    ep_loss, tr_p, tr_t = 0.0, [], []

    for step, batch in enumerate(train_loader):
        h_iids = batch['h_input_ids'].to(DEVICE)
        h_amsk = batch['h_attention_mask'].to(DEVICE)
        b_iids = batch['b_input_ids'].to(DEVICE)
        b_amsk = batch['b_attention_mask'].to(DEVICE)
        lbls = batch['label'].to(DEVICE)
        manp = batch['manip'].to(DEVICE)

        optimizer.zero_grad()
        logits, mp, anom = model(h_iids, h_amsk, b_iids, b_amsk)
        loss, lc, lm, la = mt_loss(logits, lbls, mp, manp, anom)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()

        ep_loss += loss.item()
        tr_p.extend(torch.argmax(logits,1).cpu().numpy())
        tr_t.extend(lbls.cpu().numpy())

        log_every = max(1, len(train_loader) // 5)
        if (step+1) % log_every == 0 or (step+1) == len(train_loader):
            avg = ep_loss/(step+1)
            lr_ = scheduler.get_last_lr()[0]
            log.info(f"  Step [{step+1:4d}/{len(train_loader)}]  "
                     f"Loss={avg:.4f} (cls={lc:.4f} manip={lm:.4f} anom={la:.4f})  "
                     f"LR={lr_:.2e}")

    tr_f1 = f1_score(tr_t, tr_p, average='binary')
    log.ok(f"Epoch {epoch} Train  → Loss={ep_loss/len(train_loader):.4f}  F1={tr_f1:.4f}")

    # ── VALIDATE ────────────────────────────────────────────────────────────
    log.info(f"EPOCH {epoch}/{EPOCHS} ── Validating …")
    model.eval()
    vl_loss, vp, vt, vprobs = 0.0, [], [], []
    vmp_list, vmt_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            h_iids = batch['h_input_ids'].to(DEVICE)
            h_amsk = batch['h_attention_mask'].to(DEVICE)
            b_iids = batch['b_input_ids'].to(DEVICE)
            b_amsk = batch['b_attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            manp = batch['manip'].to(DEVICE)
            logits, mp, anom = model(h_iids, h_amsk, b_iids, b_amsk)
            loss, *_ = mt_loss(logits, lbls, mp, manp, anom)
            vl_loss += loss.item()
            vp.extend(torch.argmax(logits,1).cpu().numpy())
            vt.extend(lbls.cpu().numpy())
            vprobs.extend(F.softmax(logits,1)[:,1].cpu().numpy())
            vmp_list.append(mp.cpu().numpy())
            vmt_list.append(manp.cpu().numpy())

    vf1  = f1_score(vt, vp, average='binary')
    vpr  = precision_score(vt, vp, average='binary', zero_division=0)
    vrc  = recall_score(vt, vp, average='binary', zero_division=0)
    try:    vauc = roc_auc_score(vt, vprobs)
    except: vauc = 0.0
    avg_vl = vl_loss/len(val_loader)

    history['train_loss'].append(ep_loss/len(train_loader))
    history['val_loss'].append(avg_vl)
    history['val_f1'].append(vf1)
    history['val_auc'].append(vauc)

    log.metric(f"Epoch {epoch} Val → Loss={avg_vl:.4f}  F1={vf1:.4f}  "
               f"Prec={vpr:.4f}  Rec={vrc:.4f}  AUC={vauc:.4f}")

    # Per-tactic manipulation stats
    mp_all = np.vstack(vmp_list); mt_all = np.vstack(vmt_list)
    mp_bin = (mp_all >= 0.5).astype(int)
    log.info("  Manipulation head per-tactic F1:")
    for i, lbl in enumerate(MANIP_LABELS):
        mf1 = f1_score(mt_all[:,i], mp_bin[:,i], average='binary', zero_division=0)
        log.info(f"    [{lbl:12s}] F1={mf1:.3f}  triggered={mp_bin[:,i].sum():,}/{len(mp_bin):,}")

    if vf1 > best_f1:
        best_f1 = vf1
        torch.save(model.state_dict(), OUTPUT_DIR / 'best_model.pt')
        log.ok(f"  ★ Best checkpoint saved  (F1={best_f1:.4f})")
    log.sep()

log.ok(f"Training complete — Best Validation F1: {best_f1:.4f}")


=== PHASE 3 — DUAL PARALLEL ANALYSIS: Multi-Task Training ===
[INFO] Splitting: 70% train | 15% val | 15% zero-day test …
[OK] Train: 34,356  |  Val: 7,362  |  ZD-Test: 7,362
[METRIC] Train  → Phishing: 21,748  Legit: 12,608
[METRIC] ZD Test→ Phishing: 4,660  Legit: 2,702


In [ ]:
# =============================================================================
# PHASE 3B — RANDOM FOREST + XGBOOST BASELINES
# =============================================================================
log.stage("PHASE 3B — RANDOM FOREST + XGBOOST BASELINES")

rf_text_train = (train_df['header'].fillna('').astype(str) + ' ' + train_df['body'].fillna('').astype(str)).str.strip()
rf_text_val   = (val_df['header'].fillna('').astype(str) + ' ' + val_df['body'].fillna('').astype(str)).str.strip()
rf_text_test  = (test_df['header'].fillna('').astype(str) + ' ' + test_df['body'].fillna('').astype(str)).str.strip()
rf_vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, stop_words='english')
rf_x_train = rf_vectorizer.fit_transform(rf_text_train)
rf_x_val   = rf_vectorizer.transform(rf_text_val)
rf_x_test  = rf_vectorizer.transform(rf_text_test)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(rf_x_train, train_df['label'].values)
rf_val_proba = rf_model.predict_proba(rf_x_val)[:, 1]
rf_val_pred = (rf_val_proba >= 0.5).astype(int)
rf_test_proba = rf_model.predict_proba(rf_x_test)[:, 1]
rf_test_pred = (rf_test_proba >= 0.5).astype(int)
rf_val_f1 = f1_score(val_df['label'].values, rf_val_pred, average='binary')
rf_val_auc = roc_auc_score(val_df['label'].values, rf_val_proba)
rf_val_acc = accuracy_score(val_df['label'].values, rf_val_pred)
rf_test_f1 = f1_score(test_df['label'].values, rf_test_pred, average='binary')
rf_test_auc = roc_auc_score(test_df['label'].values, rf_test_proba)
rf_test_acc = accuracy_score(test_df['label'].values, rf_test_pred)
joblib.dump(rf_model, OUTPUT_DIR / 'rf_model.joblib')
joblib.dump(rf_vectorizer, OUTPUT_DIR / 'tfidf_vectorizer.joblib')
log.metric(f"Random Forest baseline → Val F1={rf_val_f1:.4f}  AUC={rf_val_auc:.4f}")
log.metric(f"Random Forest baseline → Test F1={rf_test_f1:.4f}  AUC={rf_test_auc:.4f}")

xgb_scale_pos_weight = max(1.0, float((train_df['label'] == 0).sum()) / max(1, float(train_df['label'].sum())))
xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    min_child_weight=1,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scale_pos_weight=xgb_scale_pos_weight,
)
xgb_model.fit(rf_x_train, train_df['label'].values)
xgb_val_proba = xgb_model.predict_proba(rf_x_val)[:, 1]
xgb_val_pred = (xgb_val_proba >= 0.5).astype(int)
xgb_test_proba = xgb_model.predict_proba(rf_x_test)[:, 1]
xgb_test_pred = (xgb_test_proba >= 0.5).astype(int)
xgb_val_f1 = f1_score(val_df['label'].values, xgb_val_pred, average='binary')
xgb_val_auc = roc_auc_score(val_df['label'].values, xgb_val_proba)
xgb_val_acc = accuracy_score(val_df['label'].values, xgb_val_pred)
xgb_test_f1 = f1_score(test_df['label'].values, xgb_test_pred, average='binary')
xgb_test_auc = roc_auc_score(test_df['label'].values, xgb_test_proba)
xgb_test_acc = accuracy_score(test_df['label'].values, xgb_test_pred)
joblib.dump(xgb_model, OUTPUT_DIR / 'xgb_model.joblib')
log.metric(f"XGBoost baseline → Val F1={xgb_val_f1:.4f}  AUC={xgb_val_auc:.4f}")
log.metric(f"XGBoost baseline → Test F1={xgb_test_f1:.4f}  AUC={xgb_test_auc:.4f}")

comparison_rows = [
    {
        'model': 'RandomForest',
        'val_f1': rf_val_f1,
        'val_auc': rf_val_auc,
        'val_acc': rf_val_acc,
        'test_f1': rf_test_f1,
        'test_auc': rf_test_auc,
        'test_acc': rf_test_acc,
        'artifact': str(OUTPUT_DIR / 'rf_model.joblib'),
        'vectorizer': str(OUTPUT_DIR / 'tfidf_vectorizer.joblib'),
        'feature_type': 'tfidf_combined',
    },
    {
        'model': 'XGBoost',
        'val_f1': xgb_val_f1,
        'val_auc': xgb_val_auc,
        'val_acc': xgb_val_acc,
        'test_f1': xgb_test_f1,
        'test_auc': xgb_test_auc,
        'test_acc': xgb_test_acc,
        'artifact': str(OUTPUT_DIR / 'xgb_model.joblib'),
        'vectorizer': str(OUTPUT_DIR / 'tfidf_vectorizer.joblib'),
        'feature_type': 'tfidf_combined',
    },
]



=== PHASE 3B — RANDOM FOREST + XGBOOST BASELINES ===
[METRIC] Random Forest baseline → Val F1=0.9907  AUC=0.9990
[METRIC] Random Forest baseline → Test F1=0.9898  AUC=0.9988
[METRIC] XGBoost baseline → Val F1=0.9897  AUC=0.9987
[METRIC] XGBoost baseline → Test F1=0.9882  AUC=0.9985


In [ ]:
comparison_rows.append({
    'model': 'DistilBERT',
    'val_f1': history['val_f1'][-1] if history['val_f1'] else best_f1,
    'val_auc': history['val_auc'][-1] if history['val_auc'] else auc,
    'val_acc': acc,
    'test_f1': f1,
    'test_auc': auc,
    'test_acc': acc,
    'artifact': str(OUTPUT_DIR / 'best_model.pt'),
    'vectorizer': '',
    'feature_type': 'dual_encoder_header_body',
})

PHASE 4 - ZERO-DAY RISK INFERENCE ENGINE


RuntimeError: Missing prerequisites. Run Cell 2 and Cell 3 first.

In [ ]:
# =============================================================================
# PHASE 5 — LLM REASONING ENGINE  (Explainable Output)
# =============================================================================
log.stage("PHASE 5 — LLM REASONING ENGINE (MITRE ATT&CK + Explainability)")

MITRE_MAP = {
    'authority': 'T1534 — Internal Spearphishing / Authority Impersonation',
    'fear':      'T1566.001 — Spearphishing / Fear-based Coercion',
    'urgency':   'T1659 — Content Injection with Urgency Pressure',
    'reward':    'T1598 — Phishing for Information via Reward Luring',
    'trust':     'T1566.002 — Spearphishing Link / Trust Establishment',
}

def explain(text, intent_s, manip_vec, anom_s, comp_s, verd):
    lines = [
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
        "  ZERO-DAY DETECTION — ACTIONABLE VERDICT",
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
        f"  Composite Risk Score : {comp_s:.3f}",
        f"  Phishing Intent Score: {intent_s:.3f}  [RoBERTa Semantic Head]",
        f"  Manipulation Score   : {np.mean(manip_vec):.3f}  [5-tactic mean]",
        f"  Anomaly Score        : {anom_s:.3f}  [Zero-Day novelty signal]",
        f"  ▶ Final Verdict      : {verd}",
        "",
        "  DETECTED MANIPULATION TACTICS (MITRE ATT&CK):",
    ]
    found = False
    for i, lbl in enumerate(MANIP_LABELS):
        if manip_vec[i] >= CFG['manip_threshold']:
            lines.append(f"    ✗ {lbl.upper():12s} score={manip_vec[i]:.3f}  {MITRE_MAP[lbl]}")
            found = True
    if not found:
        lines.append("    ✓ No strong manipulation tactics detected.")

    lines += ["", "  SUSPICIOUS PHRASES FOUND:"]
    t_low = text.lower()
    phrases = []
    for tac, kws in LEXICONS.items():
        for kw in kws:
            if kw in t_low: phrases.append(f'"{kw}" [{tac}]')
    if phrases:
        for p in phrases[:8]: lines.append(f"    ► {p}")
    else:
        lines.append("    ✓ No suspicious phrases detected.")

    lines += ["", "  USER EXPLANATION:"]
    if verd == 'BLOCK':
        lines.append("  ⚠️  HIGH RISK — Strong phishing/social engineering indicators.")
        lines.append("      Do NOT click links, reply, or provide personal information.")
        lines.append("      Report to your security team immediately.")
    elif verd == 'WARN':
        lines.append("  ⚡ CAUTION — Suspicious characteristics present.")
        lines.append("      Verify the sender via an official channel before acting.")
    else:
        lines.append("  ✅ LOW RISK — Message appears legitimate.")
    lines.append("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    return "\n".join(lines)

log.info("Generating explanations for 6 real test samples …")
sample_idx = np.random.choice(len(test_df), min(6, len(test_df)), replace=False)

for rank, idx in enumerate(sample_idx, 1):
    row      = test_df.iloc[idx]
    true_lbl = int(row['label'])
    t_str    = '🔴 PHISHING'   if true_lbl      == 1 else '🟢 LEGITIMATE'
    p_str    = '🔴 PHISHING'   if all_preds[idx] == 1 else '🟢 LEGITIMATE'
    tick     = '✔ CORRECT' if all_preds[idx] == true_lbl else '✘ WRONG'
    print(f"\n{'─'*54}")
    print(f"  Sample {rank}  | True: {t_str}  Pred: {p_str}  {tick}")
    print(f"  Source : {row.get('source','N/A')}  Channel: {row.get('channel','N/A')}")
    print(f"  Text   : {str(row['text'])[:130]}…")
    print()
    print(explain(str(row['text']), float(all_probs[idx]),
                  all_manip[idx], float(all_anom[idx]),
                  float(all_comp[idx]), all_verdicts[idx]))


In [ ]:
# =============================================================================
# PHASE 6 — RESULTS DASHBOARD + SUMMARY
# =============================================================================
log.stage("PHASE 6 — FINAL OUTPUT: RESULTS DASHBOARD & SUMMARY")

fig, axes = plt.subplots(2, 3, figsize=(19, 11))
fig.suptitle('Zero-Day Phishing & Social Engineering Detection — Results Dashboard',
             fontsize=15, fontweight='bold')
C = {'red':'#E63946','teal':'#2EC4B6','orange':'#F4A261'}

# 1 — Loss curves
ax = axes[0,0]
ep = range(1, len(history['train_loss'])+1)
ax.plot(ep, history['train_loss'], 'o-', color=C['red'],  lw=2, label='Train Loss')
ax.plot(ep, history['val_loss'],   's-', color=C['teal'], lw=2, label='Val Loss')
ax.set(title='Training & Validation Loss', xlabel='Epoch', ylabel='Loss')
ax.legend(); ax.grid(alpha=0.3)

# 2 — F1 & AUC
ax = axes[0,1]
ax.plot(ep, history['val_f1'],  'o-', color=C['red'],  lw=2, label='Val F1')
ax.plot(ep, history['val_auc'], 's-', color=C['teal'], lw=2, label='Val AUC-ROC')
ax.set(title='Validation F1 & AUC-ROC', xlabel='Epoch', ylabel='Score', ylim=[0,1.05])
ax.legend(); ax.grid(alpha=0.3)

# 3 — Confusion Matrix
ax = axes[0,2]
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Legitimate','Phishing'],
            yticklabels=['Legitimate','Phishing'], ax=ax)
ax.set(title='Confusion Matrix (Zero-Day Test)', ylabel='True', xlabel='Predicted')

# 4 — Composite risk distribution
ax = axes[1,0]
ax.hist(all_comp[all_labels==0], bins=40, alpha=0.7, color=C['teal'], label='Legitimate', density=True)
ax.hist(all_comp[all_labels==1], bins=40, alpha=0.7, color=C['red'],  label='Phishing',   density=True)
ax.axvline(CFG['risk_low'], color='gold',    ls='--', lw=2, label=f"WARN thr={CFG['risk_low']}")
ax.axvline(CFG['risk_mid'], color='crimson', ls='--', lw=2, label=f"BLOCK thr={CFG['risk_mid']}")
ax.set(title='Composite Risk Score Distribution', xlabel='Score', ylabel='Density')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 5 — Manipulation tactics
ax = axes[1,1]
mb = (all_manip >= CFG['manip_threshold']).astype(int)
x  = np.arange(5); w = 0.35
ax.bar(x-w/2, [mb[all_labels==1,i].sum() for i in range(5)],
       w, color=C['red'],  label='Phishing',   alpha=0.85)
ax.bar(x+w/2, [mb[all_labels==0,i].sum() for i in range(5)],
       w, color=C['teal'], label='Legitimate', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([l.capitalize() for l in MANIP_LABELS], rotation=15, fontsize=9)
ax.set(title='Manipulation Tactics by True Class', ylabel='Count')
ax.legend(); ax.grid(alpha=0.3, axis='y')

# 6 — Verdict pie
ax = axes[1,2]
vc       = Counter(all_verdicts)
col_map  = {'BLOCK':C['red'],'WARN':C['orange'],'ALLOW':C['teal']}
lv, sv   = list(vc.keys()), list(vc.values())
ax.pie(sv, labels=lv, colors=[col_map.get(l,'grey') for l in lv],
       autopct='%1.1f%%', startangle=90,
       textprops={'fontsize':12,'fontweight':'bold'})
ax.set_title('Final Verdict Distribution')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'results_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()
log.ok(f"Dashboard saved → {OUTPUT_DIR / 'results_dashboard.png'}")

# ── Summary Table ──────────────────────────────────────────────────────────────
log.stage("FINAL SUMMARY")
n = len(all_verdicts)
block = (np.array(all_verdicts)=='BLOCK').sum()
warn  = (np.array(all_verdicts)=='WARN').sum()
allow = (np.array(all_verdicts)=='ALLOW').sum()

summary = pd.DataFrame({
    'Metric': ['F1 Score','Accuracy','Precision','Recall','AUC-ROC','Avg Precision',
               'Best Val F1','BLOCK %','WARN %','ALLOW %'],
    'Value':  [f"{f1:.4f}",f"{acc:.4f}",f"{prec:.4f}",f"{rec:.4f}",f"{auc:.4f}",f"{ap:.4f}",
               f"{best_f1:.4f}",
               f"{100*block/n:.1f}%",f"{100*warn/n:.1f}%",f"{100*allow/n:.1f}%"],
})
display(summary)
summary.to_csv(str(OUTPUT_DIR / 'summary_metrics.csv'), index=False)

comparison_df = pd.DataFrame(comparison_rows).sort_values(by=['val_f1', 'val_auc', 'test_f1'], ascending=[False, False, False]).reset_index(drop=True)
comparison_df.to_csv(str(OUTPUT_DIR / 'model_comparison.csv'), index=False)
display(comparison_df)

fig_cmp, axes_cmp = plt.subplots(1, 2, figsize=(15, 5))
cmp_palette = {'DistilBERT': '#1f77b4', 'RandomForest': '#2ca02c', 'XGBoost': '#ff7f0e'}
plot_df = comparison_df[['model', 'val_f1', 'test_f1']].set_index('model')
plot_df.plot(kind='bar', ax=axes_cmp[0], color=[cmp_palette.get(m, '#666666') for m in plot_df.index])
axes_cmp[0].set_title('F1 Comparison (Validation vs Test)')
axes_cmp[0].set_ylabel('F1')
axes_cmp[0].set_ylim(0, 1.05)
axes_cmp[0].grid(alpha=0.3, axis='y')
axes_cmp[0].legend(['Val F1', 'Test F1'])

plot_auc_df = comparison_df[['model', 'val_auc', 'test_auc']].set_index('model')
plot_auc_df.plot(kind='bar', ax=axes_cmp[1], color=[cmp_palette.get(m, '#666666') for m in plot_auc_df.index])
axes_cmp[1].set_title('AUC Comparison (Validation vs Test)')
axes_cmp[1].set_ylabel('AUC')
axes_cmp[1].set_ylim(0, 1.05)
axes_cmp[1].grid(alpha=0.3, axis='y')
axes_cmp[1].legend(['Val AUC', 'Test AUC'])

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

best_row = comparison_df.iloc[0].to_dict()
best_info = {
    'best_model': best_row['model'],
    'best_metric': 'val_f1',
    'best_val_f1': float(best_row['val_f1']),
    'best_test_f1': float(best_row['test_f1']),
    'best_artifact': best_row['artifact'],
    'best_vectorizer': best_row.get('vectorizer', ''),
    'feature_type': best_row.get('feature_type', ''),
    'model_name': MODEL_NAME,
    'device': str(DEVICE),
}
with open(OUTPUT_DIR / 'best_model_info.json', 'w', encoding='utf-8') as f:
    json.dump(best_info, f, indent=2)
log.ok(f"Best model selected for extension app: {best_info['best_model']}")

log.sep()
log.stage("ARCHITECTURE VERIFICATION CHECKLIST")
checks = [
    ("Phase 1  Context-Aware Text Preprocessing",       "Sent-seg · NER stubs · URL/entity/lexicon features"),
    ("Phase 2A Semantic Intent Encoder (RoBERTa-base)", "12 layers · d=768 · 12 heads · 4-way pooling"),
    ("Phase 2B Input Embedding Layer",                  "Token+Position+Segment → LayerNorm → 768-dim"),
    ("Phase 2C 4-way Pooling → 3072-dim concat",        "CLS · token-mean · span-mean · SEP pooled reps"),
    ("Phase 3A Psychological Manipulation Analyser",    "Tensor Projection 768→256→5, multi-label BCE"),
    ("Phase 3B Primary Phishing Risk Classifier",       "3072→512→256→2, GELU, Dropout multi-task"),
    ("Phase 4  Zero-Day Risk Inference Engine",         "0.40·Intent + 0.35·Manip + 0.25·Anomaly fusion"),
    ("Phase 5  LLM Reasoning Engine",                   "MITRE ATT&CK · phrase highlights · verdict"),
    ("Phase 6  Final Actionable Output",                "Risk score · manip vector · BLOCK/WARN/ALLOW"),
    ("ZD Eval  Zero-Day Evaluation Strategy",           "Real held-out test · anomaly head · composite"),
    ("Data     No synthetic data",                      "ethan + naser (7 files) + uciml SMS only"),
]
for s, d in checks:
    log.ok(f"  ✔ {s:<50} {d}")

log.sep()
log.ok(f"OUTPUTS → {OUTPUT_DIR / 'best_model.pt'}")
log.ok(f"          {OUTPUT_DIR / 'results_dashboard.png'}")
log.ok(f"          {OUTPUT_DIR / 'summary_metrics.csv'}")
log.ok("ZERO-DAY PHISHING DETECTION PIPELINE COMPLETE ✓")